In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.backtest import (
    blend,
    fit_blend_weight,
    paired_brier_difference,
    pooled_brier,
    scope_to_leagues,
    split_by_date,
)
from transforms.calibration import to_long_format, fit_all_calibration_curves, calibrate_matches
from transforms.dimensions import LEAGUE_COUNTRY
from transforms.elo import build_elo
from transforms.elo_probability import apply_elo_models, to_elo_long_format
from transforms.pooling import apply_logistic_pool, coefficient_summary, fit_logistic_pool
from transforms.scoring import expected_home_score_column

SILVER_TABLE = "stryktipset.silver.matches"

# Scored on the five English tiers only -- the same scope as gold, reusing
# gold's own league list so the two cannot drift apart. Sweden, the cups
# and everything else are excluded from the fit and from the judgement.
SCOPE = tuple(LEAGUE_COUNTRY)

# Everything before this trains; this day and after is scored.
# A date rather than a percentage, so the boundary lands on a season edge
# instead of slicing a draw in half. July is the gap between English
# seasons -- the same cutoff month build_dim_season uses -- so no season is
# split across train and test.
#
# Moved from 2023-01-01 once the scope narrowed to England: dropping Sweden
# left a 72/28 split, because Sweden's share of the history is not uniform
# across the years. This lands nearer the intended 80/20.
CUTOFF = "2023-07-01"

In [ ]:
def describe(name: str, df: DataFrame) -> None:
    """Row count and date range, so the split is visible rather than assumed."""
    row = df.agg(
        F.count("*").alias("n"),
        F.min("match_start").alias("first"),
        F.max("match_start").alias("last"),
    ).first()
    print(f"{name:<6} {row['n']:>6} matches   {str(row['first'])[:10]} .. {str(row['last'])[:10]}")


def report(label: str, df: DataFrame, baseline: str, candidate: str) -> None:
    """Paired comparison of two prediction sets over the same matches."""
    stats = paired_brier_difference(df, baseline=baseline, candidate=candidate)
    verdict = (
        "better" if stats["ci_low"] > 0
        else "WORSE" if stats["ci_high"] < 0
        else "no measurable difference"
    )
    print(
        f"{label:<34}{stats['mean_difference']:+.5f}"
        f"  [{stats['ci_low']:+.5f}, {stats['ci_high']:+.5f}]"
        f"  t={stats['t_statistic']:>6.2f}   {verdict}"
    )

In [ ]:
def main():
    """Does Elo know anything the crowd doesn't?

    Elo alone will probably lose to the crowd, and that would not make it
    useless. Elo knows only results; the market also knows injuries,
    suspensions, form and motivation. The question worth asking is whether
    combining the two beats the crowd on its own -- if it does, Elo carries
    independent information and the `gap` column means something.

    Everything is fitted on matches before CUTOFF and scored on matches
    after it, so no curve has seen the results it is judged on. Elo itself
    needs no such care: build_elo records each rating before kick-off, so
    it is leak-free by construction.

    Scoped to SCOPE (the English tiers) after Elo is built, so cup results
    still inform the ratings while only English league matches are fitted
    and judged.
    """
    everything = spark.table(SILVER_TABLE)

    elo = build_elo(everything)
    everything = everything.join(elo, on="match_id", how="inner").withColumn(
        "elo_expected_score",
        expected_home_score_column(F.col("elo_home"), F.col("elo_away")),
    )
    matches = scope_to_leagues(everything, SCOPE)

    train, test = split_by_date(matches, CUTOFF)
    print(f"scope  {', '.join(SCOPE)}")
    describe("all", everything.filter(F.col("result").isNotNull()))
    print(f"cutoff {CUTOFF}")
    describe("train", train)
    describe("test", test)
    played = train.count() + test.count()
    print(f"       train {train.count() / played:.0%} / test {test.count() / played:.0%}")
    print()

    crowd_models = fit_all_calibration_curves(to_long_format(train))
    elo_models = fit_all_calibration_curves(to_elo_long_format(train))

    scored = apply_elo_models(calibrate_matches(test, crowd_models), elo_models)

    # An equal-weight average is a blunt test: blending a clearly weaker
    # forecast at half weight hurts almost regardless, so it measures the
    # choice of weight as much as the forecast. Fit the weight on the
    # TRAINING half instead and ask whether *any* amount of Elo helps.
    trained = apply_elo_models(calibrate_matches(train, crowd_models), elo_models)
    weight, _ = fit_blend_weight(trained, "calibrated", "elo")

    scored = blend(scored, "calibrated", "elo", 0.5, prefix="combined")
    scored = blend(scored, "calibrated", "elo", weight, prefix="fitted")

    # A blend can only land between its two inputs. Logistic pooling works
    # in log-odds and its coefficients need not sum to 1, so it can be more
    # confident than either -- and the interaction asks whether Elo earns
    # its keep only where the crowd has no strong opinion.
    pool = fit_logistic_pool(trained)
    scored = apply_logistic_pool(scored, pool, prefix="pooled")

    pool_x = fit_logistic_pool(trained, interaction=True)
    scored = apply_logistic_pool(scored, pool_x, interaction=True, prefix="pooledx")

    print("pooled Brier on the test period (lower is better)")
    print(f"  {'33/33/33 (knows nothing)':<26}0.2222")
    for label, prefix in [("raw streck", "streck"), ("calibrated crowd", "calibrated"),
                          ("elo alone", "elo"), ("crowd + elo, 50/50", "combined"),
                          (f"crowd + elo, fitted w={weight:.2f}", "fitted"),
                          ("logistic pool", "pooled"),
                          ("logistic pool + interaction", "pooledx")]:
        print(f"  {label:<26}{pooled_brier(scored, prefix):.4f}")

    print()
    print(f"{'paired difference':<34}{'mean':>8}  {'95% interval':^22}")
    report("calibrated vs raw streck", scored, "streck", "calibrated")
    report("elo vs calibrated crowd", scored, "calibrated", "elo")
    report("50/50 blend vs crowd alone", scored, "calibrated", "combined")
    report(f"fitted blend (w={weight:.2f}) vs crowd", scored, "calibrated", "fitted")
    report("logistic pool vs crowd", scored, "calibrated", "pooled")
    report("pool + interaction vs crowd", scored, "calibrated", "pooledx")

    print()
    print("mean |coefficient| per source -- near zero means the fit ignores it")
    print(f"  additive     {coefficient_summary(pool)}")
    print(f"  interaction  {coefficient_summary(pool_x, interaction=True)}")
    if weight == 1.0:
        print()
        print("The fitted weight is 1.00: no amount of Elo improves on the crowd.")

    combined_gain = paired_brier_difference(scored, baseline="calibrated", candidate="fitted")
    calibration_gain = paired_brier_difference(scored, baseline="streck", candidate="calibrated")
    dbutils.notebook.exit(
        f"scope=english cutoff={CUTOFF} train={train.count()} test={test.count()} "
        f"streck={pooled_brier(scored, 'streck'):.4f} "
        f"calibrated={pooled_brier(scored, 'calibrated'):.4f} "
        f"elo={pooled_brier(scored, 'elo'):.4f} "
        f"cal_gain={calibration_gain['mean_difference']:+.5f} "
        f"cal_t={calibration_gain['t_statistic']:.2f} "
        f"combined={pooled_brier(scored, 'combined'):.4f} "
        f"fitted_w={weight:.2f} fitted={pooled_brier(scored, 'fitted'):.4f} "
        f"pooled={pooled_brier(scored, 'pooled'):.4f} "
        f"pooledx={pooled_brier(scored, 'pooledx'):.4f} "
        f"coef[{coefficient_summary(pool)}] "
        f"pool_gain={paired_brier_difference(scored, 'calibrated', 'pooled')['mean_difference']:+.5f} "
        f"pool_t={paired_brier_difference(scored, 'calibrated', 'pooled')['t_statistic']:.2f} "
        f"combined_gain={combined_gain['mean_difference']:+.5f} "
        f"ci={combined_gain['ci_low']:+.5f}..{combined_gain['ci_high']:+.5f} "
        f"t={combined_gain['t_statistic']:.2f}"
    )

main()